In [ ]:
import vivarium_inputs
import gbd_mapping
import pathlib
import pandas as pd, numpy as np

In [ ]:
location = "india"

In [ ]:
total_population = vivarium_inputs.get_population_structure(location.title()).droplevel(["location", "year_start", "year_end"]).value
total_population

In [ ]:
path = pathlib.Path(f'../results/population/total/{location}.csv')
path.parent.mkdir(exist_ok=True, parents=True)
total_population.rename("value").to_csv(path)

In [ ]:
asfr = vivarium_inputs.get_measure(gbd_mapping.covariates.age_specific_fertility_rate, "estimate", location.title()).droplevel(["location", "year_start", "year_end"])
asfr = asfr[asfr.index.get_level_values('parameter') == 'mean_value'].droplevel("parameter").value
asfr[asfr > 0]

In [ ]:
sbr = vivarium_inputs.get_measure(gbd_mapping.covariates.stillbirth_to_live_birth_ratio, "estimate", location.title())
sbr = sbr[sbr.index.get_level_values('parameter') == 'mean_value'].droplevel("parameter").value
assert len(sbr) == 1
sbr = sbr.iloc[0]
sbr

In [ ]:
maternal_abortion_miscarriage_incidence = vivarium_inputs.get_measure(gbd_mapping.causes.maternal_abortion_and_miscarriage, "incidence_rate", location.title()).droplevel(["location", "year_start", "year_end"])
maternal_abortion_miscarriage_incidence = maternal_abortion_miscarriage_incidence.mean(axis=1)
maternal_abortion_miscarriage_incidence[maternal_abortion_miscarriage_incidence > 0]

In [ ]:
ectopic_pregnancy_incidence = vivarium_inputs.get_measure(gbd_mapping.causes.ectopic_pregnancy, "incidence_rate", location.title()).droplevel(["location", "year_start", "year_end"])
ectopic_pregnancy_incidence = ectopic_pregnancy_incidence.mean(axis=1)
ectopic_pregnancy_incidence[ectopic_pregnancy_incidence > 0]

In [ ]:
pregnancy_incidence = (
    asfr
    + (asfr * sbr)
    + maternal_abortion_miscarriage_incidence
    + ectopic_pregnancy_incidence
)
pregnancy_incidence[pregnancy_incidence > 0]

In [ ]:
path = pathlib.Path(f'../results/pregnancy/incidence/{location}.csv')
path.parent.mkdir(exist_ok=True, parents=True)
pregnancy_incidence.rename("value").to_csv(path)

In [ ]:
# https://github.com/ihmeuw/vivarium_gates_iv_iron/blob/862b057457cea2410e8d24c2c9c38c43419cd8b2/src/vivarium_gates_iv_iron/data/loader.py#L237C5-L240C6
PARTIAL_TERM: float = 24 / 52
FULL_TERM: float = 40 / 52
pregnancy_prevalence = (
    FULL_TERM * (asfr + asfr * sbr)
    + PARTIAL_TERM * (maternal_abortion_miscarriage_incidence + ectopic_pregnancy_incidence)
)
pregnancy_prevalence[pregnancy_prevalence > 0]

In [ ]:
path = pathlib.Path(f'../results/pregnancy/prevalence/{location}.csv')
path.parent.mkdir(exist_ok=True, parents=True)
pregnancy_prevalence.rename("value").to_csv(path)

In [ ]:
population_stratified = pd.concat([
    (pregnancy_prevalence * total_population).rename("value").to_frame().assign(pregnant="pregnant").set_index("pregnant", append=True).value,
    ((1 - pregnancy_prevalence) * total_population).rename("value").to_frame().assign(pregnant="not_pregnant").set_index("pregnant", append=True).value,
]).sort_index()
population_stratified

In [ ]:
assert np.isclose(population_stratified.sum(), total_population.sum())

In [ ]:
population_age_groups = total_population.reset_index()[["age_start", "age_end"]].drop_duplicates().sort_values("age_start")
population_age_groups

In [ ]:
def map_to_population_age_groups(df):
    return (
        population_age_groups.merge(df, how="cross", suffixes=("", "_orig"))
            .pipe(lambda df: df[(df.age_end <= df.age_end_orig) & (df.age_start >= df.age_start_orig)])
            .drop(columns=["age_start_orig", "age_end_orig"])
    )

In [ ]:
wealth_quintile_probabilities = pd.read_csv(f'../results/wealth_quintile_probabilities/{location}.csv').set_index(["sex", "age_start", "age_end", "pregnant"])
wealth_quintile_probabilities.columns.name = "wealth_quintile"
wealth_quintile_probabilities = wealth_quintile_probabilities.stack()
wealth_quintile_probabilities

In [ ]:
wealth_quintile_probabilities = map_to_population_age_groups(wealth_quintile_probabilities.rename("value").reset_index()).set_index(wealth_quintile_probabilities.index.names).value
wealth_quintile_probabilities

In [ ]:
assert np.allclose(wealth_quintile_probabilities.groupby(population_stratified.index.names).sum(), 1.0)

In [ ]:
population_stratified = population_stratified.mul(wealth_quintile_probabilities, axis=0).dropna()
population_stratified

In [ ]:
population_stratified.sum()

In [ ]:
total_population.sum()

In [ ]:
assert np.isclose(population_stratified.sum(), total_population.sum(), rtol=0.01)

In [ ]:
path = pathlib.Path(f'../results/population/stratified/{location}.csv')
path.parent.mkdir(exist_ok=True, parents=True)
population_stratified.rename("value").to_csv(path)